# NVIDIA Developer Program — Exploratory Data Analysis

Tables used:
- `activity_clean` — developer engagement events and activity scores
- `contact_clean` — developer profile info (account, geography, interests, lifecycle dates)
- `contact_supplement_clean` — developer geography and industry verticals
- `sdk_download_clean` — aggregate SDK download counts by product and region
- `activity_score_mapping` — scoring weights by activity type (reference)


## 0. Setup & Connect

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

con = duckdb.connect("developer_project.duckdb")

In [2]:
display(con.execute("SHOW TABLES").fetchdf())

,name


---
## 1. Table-Level Overview

In [3]:
tables = ['activity_clean', 'contact_clean', 'contact_supplement_clean', 'sdk_download_clean']

for tbl in tables:
    n_rows = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    cols = con.execute(f"""
        SELECT column_name FROM information_schema.columns
        WHERE table_name = '{tbl}'
    """).fetchdf()['column_name'].tolist()
    print(f"\n{tbl}:  {n_rows:,} rows | {len(cols)} cols")
    print(" ", cols)

CatalogException: Catalog Error: Table with name activity_clean does not exist!
Did you mean "sqlite_temp_schema"?

LINE 1: SELECT COUNT(*) FROM activity_clean
                             ^

In [ ]:
# SUMMARIZE gives min, max, avg, null count for every column
for tbl in tables:
    print(f"\n{'='*60}\nSUMMARIZE: {tbl}\n{'='*60}")
    display(con.execute(f"SUMMARIZE {tbl}").fetchdf())

---
## 2. Data Quality Checks

### 2a. Duplicate rows

In [ ]:
for tbl in tables:
    display(con.execute(f"""
        SELECT
            '{tbl}'  AS table_name,
            COUNT(*) AS total_rows,
            COUNT(*) - (
                SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {tbl})
            ) AS duplicate_rows
        FROM {tbl}
    """).fetchdf())

### 2b. Duplicate developer_id in contact_clean

In [ ]:
display(con.execute("""
    SELECT
        COUNT(*)                                AS total_rows,
        COUNT(DISTINCT developer_id)            AS unique_developer_ids,
        COUNT(*) - COUNT(DISTINCT developer_id) AS duplicate_ids
    FROM contact_clean
""").fetchdf())

### 2c. Orphaned activity rows (no matching developer in contact_clean)

In [ ]:
display(con.execute("""
    SELECT COUNT(*) AS orphaned_activity_rows
    FROM activity_clean a
    LEFT JOIN contact_clean c ON a.dev_contact = c.developer_id
    WHERE c.developer_id IS NULL
""").fetchdf())

---
## 3. Activity Table Analysis

### 3a. Activity type counts

In [ ]:
display(con.execute("""
    SELECT activity_type, COUNT(*) AS count
    FROM activity_clean
    GROUP BY activity_type
    ORDER BY count DESC
""").fetchdf())

In [ ]:
act_type = con.execute("""
    SELECT activity_type, COUNT(*) AS count
    FROM activity_clean
    WHERE activity_type IS NOT NULL
    GROUP BY activity_type
    ORDER BY count DESC
""").fetchdf()

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=act_type, x='count', y='activity_type', ax=ax)
ax.set_title('Activity Volume by Type')
ax.set_xlabel('Event Count')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 3b. Top activity names

In [ ]:
display(con.execute("""
    SELECT activity_name, COUNT(*) AS count
    FROM activity_clean
    GROUP BY activity_name
    ORDER BY count DESC
    LIMIT 20
""").fetchdf())

### 3c. Activity score distribution

In [ ]:
display(con.execute("""
    SELECT activity_score, COUNT(*) AS count
    FROM activity_clean
    GROUP BY activity_score
    ORDER BY count DESC
""").fetchdf())

In [ ]:
scores = con.execute("""
    SELECT activity_score
    FROM activity_clean
    WHERE activity_score IS NOT NULL
""").fetchdf()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(scores['activity_score'], bins=40, edgecolor='white')
axes[0].set_title('Activity Score Distribution (raw)')
axes[0].set_xlabel('Activity Score')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(scores['activity_score']), bins=40, edgecolor='white', color='coral')
axes[1].set_title('Activity Score Distribution (log1p scale)')
axes[1].set_xlabel('log(1 + Activity Score)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

### 3d. Average score per activity type

In [ ]:
score_by_type = con.execute("""
    SELECT
        activity_type,
        ROUND(AVG(activity_score), 2) AS avg_score,
        ROUND(SUM(activity_score), 0) AS total_score,
        COUNT(*)                      AS event_count
    FROM activity_clean
    WHERE activity_type IS NOT NULL AND activity_score IS NOT NULL
    GROUP BY activity_type
    ORDER BY avg_score DESC
""").fetchdf()

display(score_by_type)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=score_by_type, x='avg_score', y='activity_type', ax=ax, palette='viridis')
ax.set_title('Average Activity Score by Type')
ax.set_xlabel('Avg Score')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

### 3e. Per-developer engagement totals

In [ ]:
display(con.execute("""
    SELECT dev_contact, COUNT(*) AS count
    FROM activity_clean
    GROUP BY dev_contact
    ORDER BY count DESC
    LIMIT 10
""").fetchdf())

In [ ]:
dev_totals = con.execute("""
    SELECT
        dev_contact,
        SUM(activity_score)           AS total_score,
        COUNT(*)                      AS total_events,
        COUNT(DISTINCT activity_type) AS distinct_types
    FROM activity_clean
    GROUP BY dev_contact
""").fetchdf()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(np.log1p(dev_totals['total_score']), bins=60, edgecolor='white')
axes[0].set_title('Total Score per Developer (log1p)')
axes[0].set_xlabel('log(1 + Total Score)')

axes[1].hist(np.log1p(dev_totals['total_events']), bins=60, edgecolor='white', color='steelblue')
axes[1].set_title('Total Events per Developer (log1p)')
axes[1].set_xlabel('log(1 + Event Count)')

axes[2].hist(dev_totals['distinct_types'], bins=range(1, 20), edgecolor='white', color='mediumseagreen')
axes[2].set_title('Distinct Activity Types per Developer')
axes[2].set_xlabel('Distinct Types')

plt.tight_layout()
plt.show()

print(dev_totals[['total_score', 'total_events', 'distinct_types']].describe())

### 3f. Engagement breadth vs. depth

In [ ]:
sample = dev_totals.sample(min(5000, len(dev_totals)), random_state=42)

fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(
    np.log1p(sample['total_events']),
    sample['distinct_types'],
    alpha=0.25, s=15,
    c=np.log1p(sample['total_score']), cmap='plasma'
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('log(1 + Total Score)')
ax.set_title('Engagement Breadth vs. Depth (sample up to 5k devs)')
ax.set_xlabel('log(1 + Total Events)  <- Depth')
ax.set_ylabel('Distinct Activity Types  <- Breadth')
plt.tight_layout()
plt.show()

### 3g. Activity trend over time

In [ ]:
act_trend = con.execute("""
    SELECT
        DATE_TRUNC('month', activity_date) AS month,
        activity_type,
        COUNT(*)                           AS events
    FROM activity_clean
    WHERE activity_date IS NOT NULL
    GROUP BY 1, 2
    ORDER BY 1
""").fetchdf()

top6_types = act_type['activity_type'].head(6).tolist()
trend_filtered = act_trend[act_trend['activity_type'].isin(top6_types)]

fig, ax = plt.subplots(figsize=(14, 6))
for atype, grp in trend_filtered.groupby('activity_type'):
    ax.plot(grp['month'], grp['events'], label=atype, lw=1.5, marker='o', markersize=3)
ax.set_title('Monthly Activity Trend (Top 6 Types)')
ax.set_xlabel('Month')
ax.set_ylabel('Events')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(loc='upper left', fontsize=8)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## 4. Contact Table Analysis

### 4a. Developer growth over time

In [ ]:
growth = con.execute("""
    SELECT
        DATE_TRUNC('month', created_date) AS month,
        COUNT(*)                          AS new_developers
    FROM contact_clean
    WHERE created_date IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchdf()

fig, ax = plt.subplots()
ax.fill_between(growth['month'], growth['new_developers'], alpha=0.4)
ax.plot(growth['month'], growth['new_developers'], lw=1.5)
ax.set_title('New Developer Registrations by Month')
ax.set_xlabel('Month')
ax.set_ylabel('New Developers')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### 4b. RDP membership split

In [ ]:
rdp = con.execute("""
    SELECT
        CASE WHEN first_program_application_date IS NOT NULL THEN 'RDP Member'
             ELSE 'Non-RDP'
        END AS rdp_status,
        COUNT(*) AS developers
    FROM contact_clean
    GROUP BY 1
""").fetchdf()

display(rdp)

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(rdp['developers'], labels=rdp['rdp_status'], autopct='%1.1f%%', startangle=90)
ax.set_title('RDP Membership Split')
plt.tight_layout()
plt.show()

### 4c. Top program application sources

In [ ]:
sources = con.execute("""
    SELECT
        program_application_source,
        COUNT(*) AS developers
    FROM contact_clean
    WHERE program_application_source IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 15
""").fetchdf()

display(sources)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=sources, x='developers', y='program_application_source', ax=ax)
ax.set_title('Top 15 Program Application Sources')
ax.set_xlabel('Developer Count')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 4d. Account type distribution (multi-select expanded)

In [ ]:
acct_type = con.execute("""
    SELECT
        TRIM(type_value) AS account_type,
        COUNT(*)         AS developers
    FROM (
        SELECT UNNEST(STRING_SPLIT(account_type, ',')) AS type_value
        FROM contact_clean
        WHERE account_type IS NOT NULL
    )
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchdf()

display(acct_type)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=acct_type, x='account_type', y='developers', ax=ax)
ax.set_title('Developers by Account Type')
ax.set_xlabel('Account Type')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 4e. DevZone last login recency

In [ ]:
recency = con.execute("""
    SELECT
        DATEDIFF('day', devzone_last_login_date, CURRENT_DATE) AS days_since_login
    FROM contact_clean
    WHERE devzone_last_login_date IS NOT NULL
""").fetchdf()

fig, ax = plt.subplots()
ax.hist(recency['days_since_login'].clip(0, 730), bins=60, edgecolor='white', color='tomato')
ax.set_title('Days Since Last DevZone Login (capped at 730 days)')
ax.set_xlabel('Days Since Login')
ax.set_ylabel('Developers')
plt.tight_layout()
plt.show()

print(recency['days_since_login'].describe())

---
## 5. Contact Supplement — Geography & Industry

### 5a. Top 20 countries

In [ ]:
countries = con.execute("""
    SELECT
        country,
        COUNT(DISTINCT developer_id) AS developers
    FROM contact_supplement_clean
    WHERE country IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 20
""").fetchdf()

display(countries)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=countries, x='developers', y='country', ax=ax)
ax.set_title('Top 20 Countries by Developer Count')
ax.set_xlabel('Developers')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 5b. Developers by region

In [ ]:
regions = con.execute("""
    SELECT
        region,
        COUNT(DISTINCT developer_id) AS developers
    FROM contact_supplement_clean
    WHERE region IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchdf()

display(regions)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=regions, x='developers', y='region', ax=ax)
ax.set_title('Developers by Region')
ax.set_xlabel('Developers')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 5c. Top 20 industry verticals

In [ ]:
industry = con.execute("""
    SELECT
        industry_segment_vertical,
        COUNT(DISTINCT developer_id) AS developers
    FROM contact_supplement_clean
    WHERE industry_segment_vertical IS NOT NULL
      AND industry_segment_vertical != ''
    GROUP BY 1
    ORDER BY 2 DESC
    LIMIT 20
""").fetchdf()

display(industry)

fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=industry, x='developers', y='industry_segment_vertical', ax=ax)
ax.set_title('Top 20 Industry Verticals')
ax.set_xlabel('Developers')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

---
## 6. Cross-Table Insights

### 6a. Avg engagement score by region

In [ ]:
region_score = con.execute("""
    SELECT
        s.region,
        COUNT(DISTINCT c.developer_id)         AS developers,
        ROUND(AVG(dev_totals.total_score), 1)  AS avg_total_score,
        ROUND(AVG(dev_totals.total_events), 1) AS avg_events
    FROM contact_clean c
    JOIN contact_supplement_clean s ON c.developer_id = s.developer_id
    JOIN (
        SELECT dev_contact,
               SUM(activity_score) AS total_score,
               COUNT(*)            AS total_events
        FROM activity_clean
        GROUP BY dev_contact
    ) dev_totals ON c.developer_id = dev_totals.dev_contact
    WHERE s.region IS NOT NULL
    GROUP BY 1
    ORDER BY avg_total_score DESC
""").fetchdf()

display(region_score)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=region_score, x='avg_total_score', y='region', ax=axes[0])
axes[0].set_title('Avg Total Score by Region')
axes[0].set_xlabel('Avg Total Score')
axes[0].set_ylabel('')

sns.barplot(data=region_score, x='avg_events', y='region', ax=axes[1], color='steelblue')
axes[1].set_title('Avg Events by Region')
axes[1].set_xlabel('Avg Events')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

### 6b. RDP vs. non-RDP engagement

In [ ]:
rdp_eng = con.execute("""
    SELECT
        CASE WHEN c.first_program_application_date IS NOT NULL THEN 'RDP Member'
             ELSE 'Non-RDP'
        END AS rdp_status,
        COUNT(DISTINCT c.developer_id)           AS developers,
        ROUND(AVG(dev_totals.total_score), 1)    AS avg_total_score,
        ROUND(AVG(dev_totals.total_events), 1)   AS avg_events,
        ROUND(AVG(dev_totals.distinct_types), 1) AS avg_distinct_types
    FROM contact_clean c
    JOIN (
        SELECT dev_contact,
               SUM(activity_score)           AS total_score,
               COUNT(*)                      AS total_events,
               COUNT(DISTINCT activity_type) AS distinct_types
        FROM activity_clean
        GROUP BY dev_contact
    ) dev_totals ON c.developer_id = dev_totals.dev_contact
    GROUP BY 1
""").fetchdf()

display(rdp_eng)

### 6c. Avg engagement by account type

In [ ]:
acct_eng = con.execute("""
    SELECT
        TRIM(type_value)                      AS account_type,
        COUNT(DISTINCT c.developer_id)        AS developers,
        ROUND(AVG(dev_totals.total_score), 1) AS avg_total_score
    FROM (
        SELECT developer_id,
               UNNEST(STRING_SPLIT(account_type, ',')) AS type_value
        FROM contact_clean
        WHERE account_type IS NOT NULL
    ) c
    JOIN (
        SELECT dev_contact, SUM(activity_score) AS total_score
        FROM activity_clean
        GROUP BY dev_contact
    ) dev_totals ON c.developer_id = dev_totals.dev_contact
    GROUP BY 1
    ORDER BY avg_total_score DESC
""").fetchdf()

display(acct_eng)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=acct_eng, x='account_type', y='avg_total_score', ax=ax)
ax.set_title('Average Total Score by Account Type')
ax.set_xlabel('Account Type')
ax.set_ylabel('Avg Total Score')
plt.tight_layout()
plt.show()

### 6d. Top 10 most engaged developers (champion detection)

In [ ]:
display(con.execute("""
    SELECT
        a.dev_contact                      AS developer_id,
        c.normalized_account_name,
        s.country,
        s.region,
        SUM(a.activity_score)             AS total_score,
        COUNT(*)                          AS total_events,
        COUNT(DISTINCT a.activity_type)   AS distinct_types,
        MIN(a.activity_date)              AS first_activity,
        MAX(a.activity_date)              AS last_activity
    FROM activity_clean a
    JOIN contact_clean c ON a.dev_contact = c.developer_id
    LEFT JOIN contact_supplement_clean s ON a.dev_contact = s.developer_id
    GROUP BY 1, 2, 3, 4
    ORDER BY total_score DESC
    LIMIT 10
""").fetchdf())

### 6e. Cohort retention heatmap

In [ ]:
cohort = con.execute("""
    WITH base AS (
        SELECT
            c.developer_id,
            DATE_TRUNC('month', c.created_date)  AS cohort_month,
            DATE_TRUNC('month', a.activity_date) AS act_month
        FROM contact_clean c
        JOIN activity_clean a ON c.developer_id = a.dev_contact
        WHERE c.created_date IS NOT NULL AND a.activity_date IS NOT NULL
    ),
    sizes AS (
        SELECT cohort_month, COUNT(DISTINCT developer_id) AS cohort_size
        FROM base GROUP BY cohort_month
    ),
    monthly AS (
        SELECT
            cohort_month,
            DATEDIFF('month', cohort_month, act_month) AS months_after,
            COUNT(DISTINCT developer_id) AS active
        FROM base
        WHERE DATEDIFF('month', cohort_month, act_month) BETWEEN 0 AND 11
        GROUP BY 1, 2
    )
    SELECT
        m.cohort_month,
        m.months_after,
        ROUND(m.active * 100.0 / s.cohort_size, 1) AS retention_pct
    FROM monthly m
    JOIN sizes s ON m.cohort_month = s.cohort_month
    ORDER BY 1, 2
""").fetchdf()

pivot = cohort.pivot(index='cohort_month', columns='months_after', values='retention_pct')

fig, ax = plt.subplots(figsize=(14, max(6, len(pivot) * 0.35)))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlGn',
            linewidths=0.3, cbar_kws={'label': 'Retention %'}, ax=ax)
ax.set_title('Monthly Cohort Retention (% active in month N after signup)')
ax.set_xlabel('Months After Signup')
ax.set_ylabel('Cohort Month')
plt.tight_layout()
plt.show()

---
## 7. SDK Download Analysis

### 7a. Preview and columns

In [ ]:
display(con.execute("SUMMARIZE sdk_download_clean").fetchdf())

In [ ]:
display(con.execute("""
    SELECT column_name
    FROM information_schema.columns
    WHERE table_name = 'sdk_download_raw'
""").fetchdf())

### 7b. Duplicate check

In [ ]:
# Check duplicates
display(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - (
            SELECT COUNT(*) FROM (SELECT DISTINCT * FROM sdk_download_clean)
        ) AS duplicate_rows
    FROM sdk_download_clean
""").fetchdf())

### 7c. Downloads by source
*(Review column names from 7a and adjust below if needed)*

In [ ]:
sdk_by_source = con.execute("""
    SELECT
        lead_source,
        SUM(download_count) AS total_downloads
    FROM sdk_download_clean
    WHERE kpi = 1
      AND nv_flag = 0
      AND lead_source IS NOT NULL
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchdf()

display(sdk_by_source)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=sdk_by_source, x='total_downloads', y='lead_source', ax=ax, palette='viridis')
ax.set_title('SDK Downloads by Source (KPI=1, external only)')
ax.set_xlabel('Total Downloads')
ax.set_ylabel('')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 7d. SDK download trend over time

In [ ]:
sdk_trend = con.execute("""
    SELECT
        DATE_TRUNC('month', download_date) AS month,
        lead_source,
        SUM(download_count)               AS downloads
    FROM sdk_download_clean
    WHERE kpi = 1
      AND nv_flag = 0
      AND download_date IS NOT NULL
    GROUP BY 1, 2
    ORDER BY 1
""").fetchdf()

fig, ax = plt.subplots(figsize=(14, 6))
for src, grp in sdk_trend.groupby('lead_source'):
    ax.plot(grp['month'], grp['downloads'], label=src, lw=1.5, marker='o', markersize=3)
ax.set_title('SDK Downloads by Source Over Time')
ax.set_xlabel('Month')
ax.set_ylabel('Downloads')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=8, loc='upper left')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

---
## 8. Activity Score Mapping Reference

In [ ]:
try:
    score_map = con.execute("""
        SELECT * FROM activity_score_mapping
        ORDER BY score DESC
    """).fetchdf()
    display(score_map)

    fig, ax = plt.subplots(figsize=(10, max(5, len(score_map) * 0.35)))
    sns.barplot(data=score_map, x='score', y='value', ax=ax, palette='magma')
    ax.set_title('Activity Score Mapping')
    ax.set_xlabel('Score')
    ax.set_ylabel('Activity Value')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'activity_score_mapping not found or column names differ: {e}')

---
## 9. EDA Summary

In [ ]:
total_devs   = con.execute("SELECT COUNT(DISTINCT developer_id) FROM contact_clean").fetchone()[0]
total_events = con.execute("SELECT COUNT(*) FROM activity_clean").fetchone()[0]
rdp_count    = con.execute(
    "SELECT COUNT(*) FROM contact_clean WHERE first_program_application_date IS NOT NULL"
).fetchone()[0]
top_act = con.execute(
    "SELECT activity_type FROM activity_clean GROUP BY activity_type ORDER BY COUNT(*) DESC LIMIT 1"
).fetchone()[0]
top_country = con.execute(
    "SELECT country FROM contact_supplement_clean WHERE country IS NOT NULL GROUP BY country ORDER BY COUNT(*) DESC LIMIT 1"
).fetchone()[0]

print("=" * 55)
print("  EDA SUMMARY")
print("=" * 55)
print(f"  Total developers:          {total_devs:>10,}")
print(f"  Total activity events:     {total_events:>10,}")
print(f"  RDP members:               {rdp_count:>10,}  ({rdp_count/total_devs*100:.1f}%)")
print(f"  Most common activity type: {top_act}")
print(f"  Top country:               {top_country}")
print("=" * 55)
print()
print("Suggested next steps:")
print("  1. Feature engineering -- learn/build/deploy/community sub-scores")
print("  2. Tenure normalization -- scale scores by account age")
print("  3. Developer segmentation -- UMAP + HDBSCAN")
print("  4. Sequence modeling -- developer journey states")
print("  5. Account-level graph -- champion and team spillover detection")

In [ ]:
con.close()
print('Connection closed.')